# Extra check 4 - Direct `<unk>`: which English word triggers it?

Earlier results link `<unk>` to rare words, but that is only an association. Here we test it more directly: if one English word is removed from the sentence, does the model stop wanting to write `<unk>`?

For every sentence where the model writes `<unk>` (text input, same model and pinned revision, greedy, FP16): (1) find the step where `<unk>` starts; (2) score each word by removing it, feeding the same output prefix and measuring how much the probability of starting `<unk>` at that step drops; (3) translate again without the top-scoring word and, as a control, without one random other word. If `<unk>` disappears much more often in the first case, that word matters. Removing a word changes the meaning, so this shows what the model reacts to, not a fix. It is a post-hoc check; the original 4,200 predictions are not changed.

Input: `merged_3system_with_metrics.csv` in your Google Drive folder (default `MyDrive/cs760`). Results are saved to a subfolder of it every 25 sentences, so a disconnected run can be resumed; the GPU is released at the end; smoke runs write to separate `_SMOKE` files.

In [ ]:
%pip install -q "transformers>=4.40,<5" sentencepiece protobuf pandas tqdm

In [ ]:
import os, json, re, time
from pathlib import Path

USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/cs760"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path(DRIVE_DIR)
else:
    DATA_DIR = Path(".")

import numpy as np, pandas as pd, torch, transformers
from tqdm.auto import tqdm
from transformers import AutoProcessor, SeamlessM4Tv2ForTextToText

MODEL_ID = "facebook/seamless-m4t-v2-large"
MODEL_REVISION = "5f8cc790b19fc3f67a61c105133b20b34e3dcb76"   # use the same one as the main Direct run
TARGET_LANGUAGE, SOURCE_LANGUAGE = "cmn", "eng"
GENERATION_KWARGS = {"num_beams": 1, "do_sample": False, "max_new_tokens": 256}   # keep the same as main run

MERGED_CSV = DATA_DIR / "merged_3system_with_metrics.csv"
OUT_DIR = DATA_DIR / "occlusion"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 760
MAX_MINUTES = 20
AUTO_RELEASE_RUNTIME = True    # only if USE_DRIVE is True
# True: run only 5 sentences, as a quick test
SMOKE = False

# smoke runs write to separate files
SUFFIX = "_SMOKE" if SMOKE else ""
WORDS_CSV, SENT_CSV = OUT_DIR / f"occlusion_words{SUFFIX}.csv", OUT_DIR / f"occlusion_sentences{SUFFIX}.csv"

CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if CUDA else "cpu")
DTYPE = torch.float16 if CUDA else torch.float32
print("device:", DEVICE, "| dtype:", DTYPE, "| transformers:", transformers.__version__)
if not CUDA:
    print("WARNING: no GPU. Slow, and not identical to the T4/FP16 run.")

In [ ]:
df = pd.read_csv(MERGED_CSV)
assert len(df) == 4200
# unique sentences where the original speech output had a visible <unk>, fixed ids from alphabetical order
u = df.groupby("sentence").agg(speech_unk=("direct_has_unk", "max")).reset_index()
unk = u[u.speech_unk].sort_values("sentence").reset_index(drop=True)
unk["sid"] = np.arange(len(unk))
# shuffled, so any prefix is a fair sample
todo = unk.sample(frac=1, random_state=SEED).reset_index(drop=True)      
if SMOKE:
    todo = todo.head(5)
print("sentences with <unk> in the original speech output:", len(unk), "| to run now:", len(todo))

words_rows = pd.read_csv(WORDS_CSV).to_dict("records") if WORDS_CSV.exists() else []
sent_rows = pd.read_csv(SENT_CSV).to_dict("records") if SENT_CSV.exists() else []
done = {r["sid"] for r in sent_rows}
todo_left = todo[~todo.sid.isin(done)]
print("already done:", len(done), "| still to run:", len(todo_left))

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = SeamlessM4Tv2ForTextToText.from_pretrained(MODEL_ID, revision=MODEL_REVISION, dtype=DTYPE)
model.to(DEVICE).eval()
tok = processor.tokenizer
# '<' or '▁<', then 'unk' (token ids seen in the team's log and in extra check 3)
START_IDS, MID_ID = {249371, 9614}, 2105
UNK_RE = re.compile(r"<\s*unk\s*>", re.IGNORECASE)
print("start ids:", START_IDS, "| mid id:", MID_ID)

In [ ]:
@torch.inference_mode()
def translate(text):
    enc = processor(text=[text], src_lang=SOURCE_LANGUAGE, return_tensors="pt").to(DEVICE)
    seq = model.generate(**enc, tgt_lang=TARGET_LANGUAGE, **GENERATION_KWARGS)
    seq = (seq if torch.is_tensor(seq) else seq.sequences)[0].tolist()
    return seq, processor.decode(seq, skip_special_tokens=True).strip()

def find_unk_start(seq):
    for i in range(len(seq) - 1):
        if seq[i] in START_IDS and seq[i + 1] == MID_ID:
            return i
    return None

@torch.inference_mode()
def p_unk_start_after_removal(words, seq, pos):
    # variant 0 = original sentence, variant i = sentence with word i-1 removed.  Same decoder prefix seq[:pos] for all
    variants = [" ".join(words)] + [" ".join(words[:i] + words[i + 1:]) for i in range(len(words))]
    enc = processor(text=variants, src_lang=SOURCE_LANGUAGE, return_tensors="pt", padding=True).to(DEVICE)
    dec = torch.tensor([seq[:pos]] * len(variants), device=DEVICE)
    logits = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], decoder_input_ids=dec).logits[:, -1, :].float()
    probs = torch.softmax(logits, dim=-1)
    return probs[:, list(START_IDS)].sum(-1).tolist()

rng = np.random.default_rng(SEED)
def run_sentence(sentence, sid):
    words = sentence.split()
    seq, out = translate(sentence)
    pos = find_unk_start(seq)
    base = dict(sid=sid, sentence=sentence, n_words=len(words), output_original=out, has_unk_original=bool(UNK_RE.search(out)), unk_start_pos=pos)
    if pos is None or len(words) < 2:
        return base, []
    p = p_unk_start_after_removal(words, seq, pos)
    p_orig, p_rem = p[0], p[1:]
    drop = [p_orig - x for x in p_rem]
    wrows = [dict(sid=sid, word_index=i, word=w, p_unk_start_original=p_orig, p_unk_start_without=p_rem[i], importance=drop[i]) for i, w in enumerate(words)]
    top = int(np.argmax(drop))
    others = [i for i in range(len(words)) if i != top]
    rnd = int(rng.choice(others))
    _, out_top = translate(" ".join(words[:top] + words[top + 1:]))
    _, out_rnd = translate(" ".join(words[:rnd] + words[rnd + 1:]))
    base.update(p_unk_start_original=p_orig, top_word=words[top], top_importance=drop[top], random_word=words[rnd], random_importance=drop[rnd],
                output_without_top=out_top, unk_after_top_removed=bool(UNK_RE.search(out_top)),
                output_without_random=out_rnd, unk_after_random_removed=bool(UNK_RE.search(out_rnd)))
    return base, wrows

def save():
    pd.DataFrame(words_rows).to_csv(WORDS_CSV, index=False)
    pd.DataFrame(sent_rows).to_csv(SENT_CSV, index=False)

if len(todo_left):
    _ = run_sentence(todo_left.sentence.iloc[0], -1)       # warm-up run, not saved
t0 = time.time(); stopped_by = "finished all sentences"
for k, r in enumerate(tqdm(todo_left.itertuples(), total=len(todo_left)), start=1):
    s_row, w_rows = run_sentence(r.sentence, int(r.sid))
    sent_rows.append(s_row); words_rows += w_rows
    if k % 25 == 0:
        save()
    if (time.time() - t0) / 60 > MAX_MINUTES:
        stopped_by = f"time budget reached ({MAX_MINUTES} min)"; break
save()
S = pd.DataFrame(sent_rows); W = pd.DataFrame(words_rows)
print(f"sentences: {len(S)} | word rows: {len(W)} | stopped by: {stopped_by} | this session: {(time.time()-t0)/60:.1f} min")

In [ ]:
# free the GPU memory; the analysis below needs no GPU
del model
import gc; gc.collect()
if CUDA:
    torch.cuda.empty_cache()

In [ ]:
A = S[S.unk_start_pos.notna()].copy()
print("sentences that wrote <unk> again in this run:", len(A), "of", len(S))
A["unk_after_top_removed"] = A.unk_after_top_removed.astype(bool); A["unk_after_random_removed"] = A.unk_after_random_removed.astype(bool)
print()
print("Probability of starting <unk> at that step (original sentence): median", round(A.p_unk_start_original.median(), 3))
print("Drop when the TOP word is removed:   median", round(A.top_importance.median(), 3), "| share with drop > 0.2:", round((A.top_importance > .2).mean(), 3), "| > 0.5:", round((A.top_importance > .5).mean(), 3))
print("Drop when a RANDOM word is removed:  median", round(A.random_importance.median(), 3), "| share with drop > 0.2:", round((A.random_importance > .2).mean(), 3), "| > 0.5:", round((A.random_importance > .5).mean(), 3))
print()
n = len(A); a = int((~A.unk_after_top_removed).sum()); b = int((~A.unk_after_random_removed).sum())
print(f"<unk> DISAPPEARS after translating again without the top word:    {a}/{n} = {a/n:.1%}")
print(f"<unk> DISAPPEARS after translating again without a random word:   {b}/{n} = {b/n:.1%}")
ct = pd.crosstab(~A.unk_after_top_removed, ~A.unk_after_random_removed, rownames=["gone after top-word removal"], colnames=["gone after random-word removal"])
print(); print(ct)

In [ ]:
# examples
ex = A.sort_values("top_importance", ascending=False)
def show(r):
    print(f"EN : {r.sentence}\nTOP WORD: {r.top_word!r} (probability of <unk> drops by {r.top_importance:.2f})\nOUT (original)      : {r.output_original}\nOUT (without top)   : {r.output_without_top}\n")
print("=== strongest triggers ===")
for r in ex.head(8).itertuples(): show(r)
print("=== weakest (the test did NOT find a single trigger word) ===")
for r in ex.tail(6).itertuples(): show(r)

In [ ]:
summary = {
    "model_id": MODEL_ID, "revision": MODEL_REVISION, "device": str(DEVICE), "dtype": str(DTYPE),
    "generation": GENERATION_KWARGS, "seed": SEED, "smoke": SMOKE, "stopped_by": stopped_by,
    "sentences_run": int(len(S)), "sentences_with_unk_again": int(len(A)),
    "median_p_unk_start": float(A.p_unk_start_original.median()),
    "median_top_importance": float(A.top_importance.median()), "median_random_importance": float(A.random_importance.median()),
    "unk_gone_after_top_removed": int((~A.unk_after_top_removed).sum()), "unk_gone_after_random_removed": int((~A.unk_after_random_removed).sum()),
    "note": "Post-hoc diagnostic (text input). Original 4,200 Direct predictions unchanged.",
}
(OUT_DIR / f"run_summary_unk_extra_4_word_removal{SUFFIX}.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2)); print("\nResults saved to:", OUT_DIR)

In [ ]:
# release the GPU; the results are already on Google Drive
if USE_DRIVE and AUTO_RELEASE_RUNTIME:
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print("Could not release automatically:", e, "\nUse Runtime -> Disconnect and delete runtime.")
else:
    print("Remember: Runtime -> Disconnect and delete runtime, to stop using GPU quota.")